# Road Pothole Detection & Severity Analysis
## Interactive Analysis Notebook

This notebook demonstrates the complete pipeline: detect -> size -> depth -> severity.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
import cv2
import numpy as np
from main import PotholeAnalysisPipeline
from src.utils import draw_bounding_boxes
from matplotlib import pyplot as plt
%matplotlib inline

## Step 1: Initialize Pipeline
Create the analysis pipeline with detector, depth estimator, size estimator, and severity classifier.

In [ ]:
pipeline = PotholeAnalysisPipeline()
print('Pipeline initialized successfully!')

## Step 2: Load and Analyze Image
Upload a road image or specify a path to a sample image.

In [ ]:
image_path = 'data/samples/road1.jpg'
if os.path.exists(image_path):
    image = cv2.imread(image_path)
else:
    image = np.zeros((480, 640, 3), dtype=np.uint8)
    cv2.putText(image, 'Load a road image', (50, 240),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title('Input Road Image')
plt.axis('off')
plt.show()

## Step 3: Run Full Analysis
Execute detection, size estimation, depth analysis, and severity classification.

In [ ]:
result = pipeline.analyze_image(image, gps_coords=(37.7749, -122.4194))
print(f'Processing time: {result["processing_time_seconds"]:.2f}s')
print(f'Potholes detected: {len(result["detections"])}')

## Step 4: Visualize Results

In [ ]:
if result['annotated_image'] is not None:
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(result['annotated_image'], cv2.COLOR_BGR2RGB))
    plt.title('Pothole Detection & Severity Results')
    plt.axis('off')
    plt.show()

## Step 5: Detailed Detection Report

In [ ]:
report = result['report']
summary = report['severity_summary']
print(f'Critical: {summary["critical"]}')
print(f'High:     {summary["high"]}')
print(f'Medium:   {summary["medium"]}')
print(f'Low:      {summary["low"]}')
for i, p in enumerate(report['potholes'], 1):
    print(f'\nPothole #{i}:')
    print(f'  Confidence: {p["confidence"]:.4f}')
    print(f'  Size: {p.get("size_cm2", 0)} cm2 ({p.get("size_category", "N/A")})')
    print(f'  Depth: {p.get("depth_cm", 0)} cm ({p.get("depth_category", "N/A")})')
    print(f'  Severity: {p.get("severity", "Unknown")}')

## Step 6: Save Report

In [ ]:
from src.utils import save_json
output_dir = 'outputs'
os.makedirs(output_dir, exist_ok=True)
cv2.imwrite(os.path.join(output_dir, 'analysis_result.jpg'), result['annotated_image'])
save_json(report, os.path.join(output_dir, 'analysis_report.json'))
print('Results saved to outputs/')